# CREATE FLAG PARAMETER

In [0]:
dbutils.widgets.text("Incremental_flag",'0')

In [0]:
incremental_flag = dbutils.widgets.get("Incremental_flag")

In [0]:
incremental_flag

# CREATE DIMENSION model



## Fetch Relative Columns

In [0]:
df_src = spark.sql('''
                   SELECT DISTINCT(Model_ID) as Model_ID , model_category
            FROM PARQUET.`abfss://silver@azhancarstorage.dfs.core.windows.net/carsales`
''')

In [0]:
df_src.display()

# dim_model Sink - Initial and Incremental

In [0]:
if not spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    df_sink = spark.sql('''
    SELECT 1 AS dim_model_key, Model_ID, model_category
    FROM PARQUET.`abfss://silver@azhancarstorage.dfs.core.windows.net/carsales`
    WHERE 1=0
    ''')
else:
    df_sink = spark.sql('''
    SELECT dim_model_key, Model_ID, model_category
    FROM DELTA.`abfss://gold@azhancarstorage.dfs.core.windows.net/dim_model`
    
    ''')

In [0]:
df_sink.display()

### Filtering New and Old Records

In [0]:
df_filter = df_src.join(df_sink, df_src['Model_ID'] == df_sink['Model_ID'],'left')\
    .select(df_src['Model_ID'],df_src['model_category'],df_sink['dim_model_key'])
df_filter.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**df_filter_old**

In [0]:
df_filter_old = df_filter.filter(col('dim_model_key').isNotNull())
df_filter_old.display()

**df_filter_new**

In [0]:
df_filter_new = df_filter.filter(col('dim_model_key').isNull())\
                    .select(df_filter['Model_ID'],df_filter['model_category'])

df_filter_new.display()

# CREATE SURROGATE KEY

### Fetch the max surrogate key from existing table

In [0]:
if incremental_flag == '0':
    max_value = 1
else:
    max_value_df = spark.sql(''' select max('dim_model_key') from cars_catalog.gold.dim_model ''')
    max_value = max_value_df.collect()[0][0]+1

### Create Surrogate key column and ADD the max surrogate key

In [0]:
df_filter_new = df_filter_new.withColumn('dim_model_key',max_value+monotonically_increasing_id())
df_filter_new.display()

### CREATE final_df =  df_filter_new + df_filter_old

In [0]:
final_df = df_filter_old.union(df_filter_new)

In [0]:
final_df.display()

# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):

    delta_tbl = DeltaTable.forPath(spark,'abfss://gold@azhancarstorage.dfs.core.windows.net/dim_model')
    delta_tbl.alias("trg").merge(final_df.alias("src"),"trg.dim_model_key = src.dim_model_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                                .execute()


else:

      final_df.write.format('delta')\
          .mode('overwrite')\
            .option("path","abfss://gold@azhancarstorage.dfs.core.windows.net/dim_model")\
            .saveAsTable("cars_catalog.gold.dim_model")


In [0]:
%sql

SELECT * FROM cars_catalog.gold.dim_model